In [1]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.2 MB/s eta 0:00:00


In [4]:
"""
Google Colab Test Script for DataCleanser AgentGrid Agent
==========================================================

This notebook tests the DataCleanser Agent to ensure it works correctly
in Google Colab environment before deploying to AgentGrid.

Instructions:
1. Upload this file to Google Colab
2. Upload data_cleanser_agentgrid.py to /content/
3. Run all cells
"""

# ============================================================================
# CELL 1: Setup and Installations
# ============================================================================

print("=" * 70)
print("DataCleanser Agent - Google Colab Test Suite")
print("=" * 70)
print("\n📦 Installing required dependencies...\n")

# Install dependencies
!pip install pandas>=2.0.0 numpy>=1.24.0 openpyxl>=3.1.0 -q
!pip install python-dateutil>=2.8.0 phonenumbers>=8.13.0 email-validator>=2.0.0 -q
!pip install scikit-learn>=1.3.0 -q
!pip install openai>=1.0.0 -q

print("✓ All dependencies installed successfully!\n")

# ============================================================================
# CELL 2: Create Mock BaseAgent Classes (since we're testing standalone)
# ============================================================================

print("=" * 70)
print("Setting up mock AgentGrid framework...")
print("=" * 70)

from typing import Dict, Any, List, Optional
from dataclasses import dataclass

# Mock AgentGrid base classes for testing
@dataclass
class AgentInput:
    """Mock AgentInput for testing"""
    name: str
    type: str
    description: str
    required: bool = True

@dataclass
class AgentOutput:
    """Mock AgentOutput for testing"""
    name: str
    type: str
    description: str

class BaseAgent:
    """Mock BaseAgent for testing"""
    @property
    def name(self) -> str:
        raise NotImplementedError

    @property
    def description(self) -> str:
        raise NotImplementedError

    @property
    def inputs(self) -> List[AgentInput]:
        raise NotImplementedError

    @property
    def outputs(self) -> List[AgentOutput]:
        raise NotImplementedError

    def run(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        raise NotImplementedError

# Mock registry
_AGENT_REGISTRY = {}

def register_agent(agent_id: str):
    """Mock register_agent decorator"""
    def decorator(cls):
        _AGENT_REGISTRY[agent_id] = cls
        return cls
    return decorator

# Create mock app.agents modules
import sys
from types import ModuleType

# Create mock module structure
app_module = ModuleType('app')
agents_module = ModuleType('app.agents')
base_module = ModuleType('app.agents.base')
registry_module = ModuleType('app.agents.registry')

# Add mock classes to modules
base_module.BaseAgent = BaseAgent
base_module.AgentInput = AgentInput
base_module.AgentOutput = AgentOutput
registry_module.register_agent = register_agent

# Add to sys.modules
sys.modules['app'] = app_module
sys.modules['app.agents'] = agents_module
sys.modules['app.agents.base'] = base_module
sys.modules['app.agents.registry'] = registry_module

# Make them accessible from app.agents
agents_module.base = base_module
agents_module.registry = registry_module

print("✓ Mock AgentGrid framework created successfully!\n")

# ============================================================================
# CELL 3: Create Sample Test Data
# ============================================================================

print("=" * 70)
print("Creating sample test data...")
print("=" * 70)

import pandas as pd
import io

# Create sample data with various quality issues
sample_data = """customer_id,first_name,email,phone,purchase_date,amount,status,region
CUST001,JOHN,john@example.com,555-123-4567,2024-01-15,1250.50,Active,North
CUST002,jane,jane.doe@test.com,(555) 234-5678,01/20/2024,2500,ACTIVE,South
CUST003,Bob Smith,invalid-email,5553456789,2024-02-10,750,  Active  ,East
CUST004,  alice  ,alice@company.co,555.456.7890,2-15-24,1800,Pending,West
CUST005,Charlie,charlie@test,+1-555-567-8901,2024-03-01,3200.75,Pending,North
CUST006,DAVID,david@example.com,555-678-9012,03/10/2024,$950,PENDING,N/A
CUST007,emma,emma@test.com,5557890123,2024-03-20,$1100,active,South
CUST008,Frank,frank@invalid,555.456.7890,3-25-24,2750.25,Active,
CUST009,Grace,grace@example.com,555-890-1234,04/12/2024,$1500,Completed,North
CUST010,henry,henry@test.com,555.901.2345,2024-04-05,890,COMPLETED,null
CUST011,Ivy,ivy@company.co,5551234567,2024-04-18,1650.50,Pending,West
CUST012,JACK,jack@test.com,(555) 234-5678,04/18/2024,1650.50,Active,West
CUST013,karen,karen@example,555.345.6789,4-22-24,1900,active,Unknown
CUST014,mia,mia@example.com,555-567-8901,05/08/2024,$1050,ACTIVE,North
CUST015,Noah,noah@test.co,5556789012,5-20-24,$1750,Active,
CUST006,DAVID,david@example.com,555-678-9012,03/10/2024,950,PENDING,N/A
CUST047,TRACY,tracy@company.com,555-012-3456,2024-02-05,50000,pending,"""

# Save sample data
with open('/content/sample_test_data.csv', 'w') as f:
    f.write(sample_data)

print("✓ Sample test data created: /content/sample_test_data.csv")
print(f"  - 17 rows (includes duplicates)")
print(f"  - Missing values, mixed formats, outliers, etc.\n")

# ============================================================================
# CELL 4: Import and Initialize Agent
# ============================================================================

print("=" * 70)
print("Importing DataCleanser Agent...")
print("=" * 70)

import sys
sys.path.insert(0, '/content')

try:
    from data_cleanser_agentgrid import DataCleanserAgent, DATA_CLEANSER_AGENT_ID
    print("✓ DataCleanser Agent imported successfully!")
    print(f"  Agent ID: {DATA_CLEANSER_AGENT_ID}\n")
except ImportError as e:
    print("✗ Failed to import DataCleanser Agent!")
    print(f"  Error: {e}")
    print("\n⚠️  Make sure data_cleanser_agentgrid.py is uploaded to /content/")
    raise

# Initialize agent
agent = DataCleanserAgent()

print("✓ Agent initialized successfully!")
print(f"  Name: {agent.name}")
print(f"  Description: {agent.description[:100]}...\n")

# ============================================================================
# CELL 5: Test Agent Schema
# ============================================================================

print("=" * 70)
print("TEST 1: Agent Schema Validation")
print("=" * 70)

def test_schema():
    """Test that agent has proper schema"""

    errors = []

    # Test inputs
    print("\n📥 INPUT SCHEMA:")
    print("-" * 70)

    inputs = agent.inputs
    if not inputs:
        errors.append("No inputs defined")
    else:
        print(f"✓ Found {len(inputs)} input parameters:")
        for i, inp in enumerate(inputs, 1):
            required = "✓ Required" if inp.required else "  Optional"
            print(f"  {i}. {inp.name} ({inp.type}) - {required}")

            # Validate required fields
            if not inp.name:
                errors.append(f"Input {i} has no name")
            if not inp.type:
                errors.append(f"Input {i} has no type")

    # Test outputs
    print("\n📤 OUTPUT SCHEMA:")
    print("-" * 70)

    outputs = agent.outputs
    if not outputs:
        errors.append("No outputs defined")
    else:
        print(f"✓ Found {len(outputs)} output parameters:")
        for i, out in enumerate(outputs, 1):
            print(f"  {i}. {out.name} ({out.type})")

            # Validate required fields
            if not out.name:
                errors.append(f"Output {i} has no name")
            if not out.type:
                errors.append(f"Output {i} has no type")

    # Test run method
    print("\n🔧 RUN METHOD:")
    print("-" * 70)
    if hasattr(agent, 'run') and callable(agent.run):
        print("✓ run() method exists")
    else:
        errors.append("run() method not found or not callable")

    return errors

schema_errors = test_schema()

if schema_errors:
    print(f"\n✗ Schema validation failed with {len(schema_errors)} error(s):")
    for error in schema_errors:
        print(f"  - {error}")
else:
    print("\n✓ Schema validation passed!")

print("=" * 70)

# ============================================================================
# CELL 6: Test Agent Execution with Sample Data
# ============================================================================

print("\n" + "=" * 70)
print("TEST 2: Agent Execution with Sample Data")
print("=" * 70)

import base64
import json

def test_execution():
    """Test agent execution with sample data"""

    # Load sample data
    with open('/content/sample_test_data.csv', 'rb') as f:
        file_content = f.read()

    # Encode to base64
    file_data_b64 = base64.b64encode(file_content).decode()

    print(f"\n📁 Input file:")
    print(f"  Size: {len(file_content)} bytes")
    print(f"  Base64 size: {len(file_data_b64)} characters")

    # Prepare inputs
    inputs = {
        "file_data": file_data_b64,
        "file_type": "csv",
        "numeric_strategy": "median",
        "categorical_strategy": "mode",
        "duplicate_strategy": "first",
        "standardize_dates": True,
        "standardize_text": True,
        "remove_outliers": False,
        "use_ai": False  # No API key in test
    }

    print("\n⚙️  Configuration:")
    print(f"  numeric_strategy: {inputs['numeric_strategy']}")
    print(f"  categorical_strategy: {inputs['categorical_strategy']}")
    print(f"  duplicate_strategy: {inputs['duplicate_strategy']}")
    print(f"  standardize_dates: {inputs['standardize_dates']}")
    print(f"  standardize_text: {inputs['standardize_text']}")
    print(f"  remove_outliers: {inputs['remove_outliers']}")

    print("\n🚀 Executing agent...")

    try:
        # Execute agent
        result = agent.run(inputs)

        print("✓ Execution completed successfully!\n")

        # Validate output structure
        required_outputs = ['cleaned_data', 'cleaning_report', 'issues_detected', 'summary']
        missing_outputs = [o for o in required_outputs if o not in result]

        if missing_outputs:
            print(f"✗ Missing outputs: {missing_outputs}")
            return False, None

        print("✓ All required outputs present")

        return True, result

    except Exception as e:
        print(f"✗ Execution failed!")
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()
        return False, None

success, result = test_execution()

if not success:
    print("\n✗ TEST 2 FAILED")
else:
    print("\n✓ TEST 2 PASSED")

print("=" * 70)

# ============================================================================
# CELL 7: Analyze Results
# ============================================================================

if success and result:
    print("\n" + "=" * 70)
    print("TEST 3: Result Analysis")
    print("=" * 70)

    # Display summary
    summary = result['summary']
    print("\n📊 CLEANING SUMMARY:")
    print("-" * 70)
    print(f"Original Rows:         {summary['original_rows']}")
    print(f"Cleaned Rows:          {summary['cleaned_rows']}")
    print(f"Rows Removed:          {summary['rows_removed']}")
    print(f"Total Columns:         {summary['total_columns']}")
    print(f"Operations Performed:  {summary['operations_performed']}")
    print(f"Timestamp:             {summary['timestamp']}")

    # Validate summary
    summary_valid = True
    if summary['original_rows'] <= 0:
        print("  ✗ Invalid original_rows count")
        summary_valid = False
    if summary['cleaned_rows'] <= 0:
        print("  ✗ Invalid cleaned_rows count")
        summary_valid = False
    if summary['cleaned_rows'] > summary['original_rows']:
        print("  ✗ Cleaned rows cannot exceed original rows")
        summary_valid = False

    if summary_valid:
        print("  ✓ Summary values are valid")

    # Display issues detected
    issues = result['issues_detected']
    print("\n⚠️  ISSUES DETECTED:")
    print("-" * 70)

    issues_found = False

    if issues['missing_values']:
        issues_found = True
        print(f"✗ Missing Values: {len(issues['missing_values'])} columns")
        for col, info in list(issues['missing_values'].items())[:5]:
            print(f"  - {col}: {info['count']} missing ({info['percentage']:.1f}%)")

    if issues['duplicates']['exact_count'] > 0:
        issues_found = True
        print(f"✗ Duplicates: {issues['duplicates']['exact_count']} rows")

    formatting_issues = sum(len(v) for v in issues['formatting'].values())
    if formatting_issues > 0:
        issues_found = True
        print(f"✗ Formatting Issues: {formatting_issues} columns")
        for issue_type, columns in issues['formatting'].items():
            if columns:
                print(f"  - {issue_type}: {len(columns)} columns")

    if issues['data_types']:
        issues_found = True
        print(f"✗ Data Type Issues: {len(issues['data_types'])} columns")

    if issues['outliers']:
        issues_found = True
        print(f"✗ Outliers: {len(issues['outliers'])} columns")

    if not issues_found:
        print("  ✓ No major issues detected")

    # Display cleaning report
    report = result['cleaning_report']
    print(f"\n📋 CLEANING REPORT ({len(report)} operations):")
    print("-" * 70)

    for i, operation in enumerate(report[:8], 1):
        print(f"{i}. Column: {operation.get('column', 'N/A')}")
        print(f"   Issue: {operation.get('issue', 'N/A')}")
        print(f"   Action: {operation.get('action', 'N/A')}")
        if 'before' in operation and 'after' in operation:
            print(f"   Before: {operation['before']} → After: {operation['after']}")
        print()

    if len(report) > 8:
        print(f"... and {len(report) - 8} more operations\n")

    # Save cleaned data
    cleaned_data_b64 = result['cleaned_data']
    cleaned_data = base64.b64decode(cleaned_data_b64).decode()

    output_file = '/content/cleaned_data_test.csv'
    with open(output_file, 'w') as f:
        f.write(cleaned_data)

    print(f"💾 Cleaned data saved to: {output_file}")

    # Display first few rows
    print("\n📄 Cleaned Data Preview (first 5 rows):")
    print("-" * 70)
    cleaned_df = pd.read_csv(io.StringIO(cleaned_data))
    print(cleaned_df.head())

    print("\n✓ TEST 3 PASSED")
    print("=" * 70)

# ============================================================================
# CELL 8: Test Error Handling
# ============================================================================

print("\n" + "=" * 70)
print("TEST 4: Error Handling")
print("=" * 70)

def test_error_handling():
    """Test that agent handles errors properly"""

    errors_caught = []

    # Test 1: Missing required input
    print("\n1. Testing missing required input...")
    try:
        agent.run({})
        print("  ✗ Should have raised ValueError for missing file_data")
    except ValueError as e:
        print(f"  ✓ Correctly raised ValueError: {e}")
        errors_caught.append("missing_file_data")
    except Exception as e:
        print(f"  ✗ Unexpected error: {e}")

    # Test 2: Invalid file type
    print("\n2. Testing invalid file type...")
    try:
        agent.run({
            "file_data": "dGVzdA==",  # base64 "test"
            "file_type": "txt"
        })
        print("  ✗ Should have raised ValueError for invalid file_type")
    except ValueError as e:
        print(f"  ✓ Correctly raised ValueError: {e}")
        errors_caught.append("invalid_file_type")
    except Exception as e:
        print(f"  ✗ Unexpected error: {e}")

    # Test 3: Invalid base64
    print("\n3. Testing invalid base64...")
    try:
        agent.run({
            "file_data": "not-valid-base64!!!",
            "file_type": "csv"
        })
        print("  ✗ Should have raised ValueError for invalid base64")
    except ValueError as e:
        print(f"  ✓ Correctly raised ValueError: {e}")
        errors_caught.append("invalid_base64")
    except Exception as e:
        print(f"  ✗ Unexpected error: {e}")

    return len(errors_caught) == 3

error_handling_passed = test_error_handling()

if error_handling_passed:
    print("\n✓ TEST 4 PASSED - All errors handled correctly")
else:
    print("\n✗ TEST 4 FAILED - Some errors not handled properly")

print("=" * 70)

# ============================================================================
# CELL 9: Performance Test
# ============================================================================

print("\n" + "=" * 70)
print("TEST 5: Performance Test")
print("=" * 70)

import time

def test_performance():
    """Test agent performance with sample data"""

    # Load sample data
    with open('/content/sample_test_data.csv', 'rb') as f:
        file_content = f.read()

    file_data_b64 = base64.b64encode(file_content).decode()

    inputs = {
        "file_data": file_data_b64,
        "file_type": "csv",
        "numeric_strategy": "median",
        "categorical_strategy": "mode",
        "standardize_dates": True,
        "standardize_text": True,
        "use_ai": False
    }

    print("\n⏱️  Running performance test (3 iterations)...\n")

    times = []
    for i in range(3):
        start_time = time.time()
        result = agent.run(inputs)
        end_time = time.time()

        execution_time = end_time - start_time
        times.append(execution_time)
        print(f"  Iteration {i+1}: {execution_time:.3f} seconds")

    avg_time = sum(times) / len(times)
    print(f"\n  Average execution time: {avg_time:.3f} seconds")

    # Performance criteria
    performance_ok = avg_time < 10.0  # Should complete in under 10 seconds

    if performance_ok:
        print(f"  ✓ Performance is acceptable (< 10s)")
    else:
        print(f"  ⚠️  Performance is slow (> 10s)")

    return performance_ok

performance_passed = test_performance()

if performance_passed:
    print("\n✓ TEST 5 PASSED")
else:
    print("\n⚠️  TEST 5 WARNING - Performance may need optimization")

print("=" * 70)

# ============================================================================
# CELL 10: Final Summary
# ============================================================================

print("\n" + "=" * 70)
print("TEST SUMMARY")
print("=" * 70)

test_results = {
    "Schema Validation": len(schema_errors) == 0,
    "Agent Execution": success,
    "Result Analysis": success and result is not None,
    "Error Handling": error_handling_passed,
    "Performance": performance_passed
}

print("\n📊 Test Results:")
print("-" * 70)

all_passed = True
for test_name, passed in test_results.items():
    status = "✓ PASSED" if passed else "✗ FAILED"
    print(f"  {test_name:.<50} {status}")
    if not passed:
        all_passed = False

print("\n" + "=" * 70)

if all_passed:
    print("🎉 ALL TESTS PASSED!")
    print("=" * 70)
    print("\n✓ DataCleanser Agent is working correctly!")
    print("✓ Ready for deployment to AgentGrid")
    print("\nNext steps:")
    print("1. Copy data_cleanser_agentgrid.py to your AgentGrid backend")
    print("2. Run the SQL registration script")
    print("3. Add to app/agents/__init__.py")
    print("4. Restart your AgentGrid backend")
else:
    print("⚠️  SOME TESTS FAILED")
    print("=" * 70)
    print("\nPlease review the failed tests above and fix any issues.")

print("\n" + "=" * 70)
print("Test files created:")
print("  - /content/sample_test_data.csv (input)")
print("  - /content/cleaned_data_test.csv (output)")
print("=" * 70)

DataCleanser Agent - Google Colab Test Suite

📦 Installing required dependencies...



ERROR:data_cleanser_agentgrid:DataCleanser execution failed: file_data is required
ERROR:data_cleanser_agentgrid:DataCleanser execution failed: Unsupported file_type: txt. Must be csv, xlsx, xls, or json
ERROR:data_cleanser_agentgrid:DataCleanser execution failed: Invalid base64 encoded file_data: Incorrect padding


✓ All dependencies installed successfully!

Setting up mock AgentGrid framework...
✓ Mock AgentGrid framework created successfully!

Creating sample test data...
✓ Sample test data created: /content/sample_test_data.csv
  - 17 rows (includes duplicates)
  - Missing values, mixed formats, outliers, etc.

Importing DataCleanser Agent...
✓ DataCleanser Agent imported successfully!
  Agent ID: a7f3c2d1-8e4b-4a9c-b5d6-3f2e1a9c8b7d

✓ Agent initialized successfully!
  Name: DataCleanser Agent
  Description: AI-powered data cleaning agent that automatically detects and fixes data quality issues in CSV, Exce...

TEST 1: Agent Schema Validation

📥 INPUT SCHEMA:
----------------------------------------------------------------------
✓ Found 9 input parameters:
  1. file_data (string) - ✓ Required
  2. file_type (string) - ✓ Required
  3. numeric_strategy (string) -   Optional
  4. categorical_strategy (string) -   Optional
  5. duplicate_strategy (string) -   Optional
  6. standardize_dates (bool